# Country Club Case Study

In [2]:
import matplotlib.pyplot as plt
import pandas as pd

In [3]:
#Test if connection works
import sqlite3
connection = sqlite3.connect('../Data/sqlite_db_pythonsqlite.db')


### Question 10
Produce a list of facilities with a total revenue less than 1000.
The output of facility name and total revenue, sorted by revenue. Remember
that there's a different cost for guests and members!

In [18]:
Q10 = pd.read_sql_query(''' -- CTE 1: calculate cost per booking for MEMBER bookings only
                            
                            WITH mems AS 
                            (SELECT facilities.name AS name, (facilities.membercost * bookings.slots) AS price
                            FROM bookings
                            LEFT JOIN facilities USING(facid)
                            WHERE bookings.memid <> 0),
                            
                            -- CTE 2: calculate cost per booking for GUEST bookings only
                           
                            guest AS 
                            (SELECT facilities.name AS name, (facilities.guestcost * bookings.slots) AS price
                            FROM bookings
                            LEFT JOIN facilities USING(facid)
                            WHERE bookings.memid = 0)

                            -- Combine both CTEs into one list of individual booking costs,
                            -- then sum them up per facility and filter for revenue under $1000
                       
                        SELECT name, SUM(price) AS revenue
                        FROM
                        (SELECT name, price FROM mems
                        UNION ALL  -- stack member and guest bookings together (keeping duplicates)
                        SELECT name, price FROM guest) AS combined
                        GROUP BY name
                        Having revenue <1000''', connection)
Q10
                            

,name,revenue
0,Pool Table,270
1,Snooker Table,240
2,Table Tennis,180


### Question 11
Produce a report of members and who recommended them in alphabetic surname, firstname order

In [23]:
#Question 11 
Q11 = pd.read_sql_query('''SELECT CONCAT(m.surname, ' ', m.firstname) AS mem_name,
                        CONCAT(r.surname, ' ',r.firstname) AS rec_name
                        FROM members AS m
                        LEFT JOIN members AS r
                        ON m.memid = r.recommendedby''', connection)
Q11

,mem_name,rec_name
0,GUEST GUEST,
1,Smith Darren,Joplette Janice
2,Smith Darren,Butters Gerald
3,Smith Darren,Owen Charles
4,Smith Darren,Smith Jack
5,Smith Darren,Mackenzie Anna
6,Smith Tracy,Worthington-Smyth Henry
7,Smith Tracy,Purview Millicent
8,Smith Tracy,Crumpet Erica
9,Rownam Tim,Boothe Tim


### Quesiton 12
Q12: Find the facilities with their usage by member, but not guests

In [17]:
Q12 = pd.read_sql_query('''SELECT facilities.name AS facility, COUNT(bookings.bookid) AS usage
                        FROM facilities
                        LEFT JOIN bookings USING(facid)
                        WHERE bookings.memid <> 0
                        GROUP BY facilities.name''', connection)
Q12
                        
                             

,facility,usage
0,Badminton Court,344
1,Massage Room 1,421
2,Massage Room 2,27
3,Pool Table,783
4,Snooker Table,421
5,Squash Court,195
6,Table Tennis,385
7,Tennis Court 1,308
8,Tennis Court 2,276


### Question 13
Find the facilities usage by month, but not guests


In [16]:
Q13 = pd.read_sql_query('''SELECT facilities.name AS facility,
                        strftime('%m', bookings.starttime) AS month,
                        COUNT(bookings.bookid) AS usage
                        FROM facilities
                        LEFT JOIN bookings USING(facid)
                        WHERE bookings.memid <> 0
                        GROUP BY facilities.name, month''', connection)
Q13

,facility,month,usage
0,Badminton Court,07,51
1,Badminton Court,08,132
2,Badminton Court,09,161
3,Massage Room 1,07,77
4,Massage Room 1,08,153
5,Massage Room 1,09,191
6,Massage Room 2,07,4
7,Massage Room 2,08,9
8,Massage Room 2,09,14
9,Pool Table,07,103


In [18]:
connection.close()